# Multi-Modal Recipe Generation Agent
A multi-modal agentic RAG system that takes an image of ingredients or a dish (with/without a text query) and returns a detailed recipe.

**Pipeline:**  
`Image/Text` → `VLM Sandwich (CLIP-ViT + Qwen2.5-3B)` → `CLIP Text Encoder` → `FAISS Vector Search` → `Web Search Fallback` → `LLM Synthesis` → `Structured Recipe`

---


## 1. Install dependencies

In [1]:
!pip install -q langgraph langchain langchain-community \
    transformers torch torchvision accelerate \
    Pillow faiss-cpu numpy openai tavily-python \
    datasets tqdm pydantic python-dotenv \
    sentencepiece qwen-vl-utils groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


## 2. Configuration

In [ ]:
import os
import torch

os.environ["GROQ_API_KEY"] = None  # free at console.groq.com
os.environ["TAVILY_API_KEY"]= None    # for web search fallback

vlm_model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
clip_model_id = "openai/clip-vit-base-patch32"
synthesis_model_id = "llama-3.3-70b-versatile" # Lllama-3 (open-source on groq)

top_k = 5
similarity_threshold = 0.50   # if below this threshold, fall back to web search
max_fallbacks = 2

recipe_dataset_path = "corbt/all-recipes" # Recipes1M+ dataset available on Huggingface

os.makedirs("data", exist_ok=True) # local dir for storing index
faiss_index_local_path = "data/recipes.index"
dataset_limit = 1000000 # change to 1000000 for full index
batch_size = 256
recipes_metdata_local_path = "data/recipes_metadata.json"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

max_index_size = 1000500 # max flexible FAISS index (with extra room for newly inserted web content (right now supports 500 new writes)

Device: cuda


## 3. Agent State


In [3]:
from typing import TypedDict, Literal, Optional
from PIL import Image as PILImage

class AgentState(TypedDict): #Agent state stored in RAM for state context
    # For multimodal input
    image: Optional[PILImage.Image]
    text_query: Optional[str]
    combined_query: Optional[str] # used for merging intent + ingredients -> VLM
    intent_words: Optional[str]

    image_type: Optional[Literal["ingredients", "dish", "unknown"]] # either image is a completed dish or ingredients
    vlm_description: Optional[str]
    vlm_confidence: Optional[float]

    # For retrieval
    query_embedding: Optional[list]
    retrieved_recipes: Optional[list]
    retrieval_source: Optional[Literal["faiss", "web", "none"]]

    final_recipe: Optional[dict] # for final recipe

    # For general dialogue
    messages: list
    fallback_count: int
    error: Optional[str]

print("AgentState defined.")

AgentState defined.


## 4. Build FAISS Index
Downloads RecipeNLG from HuggingFace, encodes each recipe with CLIP, and saves a FAISS index to disk. (one-time offline step)

> ** Skip this cell if 'data/recipes.index' already exists. **


In [4]:
import os, json
import numpy as np
import faiss
import torch
from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor
from tqdm.auto import tqdm

print("Loading CLIP model...")
clip_model = CLIPModel.from_pretrained(clip_model_id)
clip_model.to(device)
clip_model.eval()

clip_processor = CLIPProcessor.from_pretrained(clip_model_id)

def encode_texts(texts):
    inputs = clip_processor(text=texts, return_tensors="pt", truncation=True,
                   max_length=77, padding=True)
    inputs = inputs.to(device)
    with torch.no_grad():
        embedding = clip_model.get_text_features(**inputs)
    # get_text_features returns a tensor directly, but if it returns an object, extract it
    if hasattr(embedding, "pooler_output"):
        embedding = embedding.pooler_output
    elif hasattr(embedding, "last_hidden_state"):
        embedding = embedding.last_hidden_state[:, 0, :]

    embedding = embedding / embedding.norm(dim=-1, keepdim=True)
    embedding = embedding.cpu().numpy().astype("float32")
    return embedding

print("Loading dataset...")
ds = load_dataset(recipe_dataset_path, split="train", streaming=True)

index = faiss.IndexFlatIP(512) # for exact search (increase in index size causes higher search latency)
# index = faiss.IndexHNSWFlat(512, 32)  # 32 = graph connections, for approximate search if larger index (more accurate, but more memory)
# index.hnsw.efConstruction = 200
# index.hnsw.efSearch = 128
metadata = []
batch_t = []
batch_m = []
count = 0

for recipe in tqdm(ds, total=dataset_limit, unit="recipe"):
    if count >= dataset_limit:
        break

    title = recipe.get("title") or ""
    ingredients = recipe.get("ingredients") or ""
    steps = recipe.get("directions") or recipe.get("description") or ""
    url = recipe.get("url") or ""

    # ingredients is a plain string on this dataset, not a list
    if isinstance(ingredients, str):
        ingredients = [i.strip() for i in ingredients.split("\n") if i.strip()]

    # steps is also a plain string on this dataset
    if isinstance(steps, str):
        steps = [s.strip() for s in steps.split("\n") if s.strip()]

    ingredients_str = ", ".join(ingredients[:10])
    batch_t.append(f"{title}: {ingredients_str}")
    batch_m.append({
        "title": title,
        "ingredients": ingredients,
        "steps": steps,
        "url": url,
    })
    count += 1

    if len(batch_t) == batch_size:
        index.add(encode_texts(batch_t))
        metadata.extend(batch_m)
        batch_t, batch_m = [], []

if batch_t:
    index.add(encode_texts(batch_t))
    metadata.extend(batch_m)

faiss.write_index(index, faiss_index_local_path)
with open(recipes_metdata_local_path, "w") as f:
    json.dump(metadata, f)

print(f"\nDone! {index.ntotal:,} recipes indexed.")

Loading CLIP model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Loading dataset...


README.md:   0%|          | 0.00/451 [00:00<?, ?B/s]

  0%|          | 0/1000000 [00:00<?, ?recipe/s]


Done! 1,000,000 recipes indexed.


## 5. Vision Module — CLIP-ViT + Qwen2.5-3B VLM
**CLIP-ViT** encodes the image into a 512-dim embedding.  
**Qwen2.5-3B** classifies it as Type A (ingredients) or Type B (dish) and extracts a text description.


In [5]:
import re
import torch
import numpy as np
from PIL import Image as PILImage
from transformers import CLIPModel, CLIPProcessor
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info

vlm_system_prompt = """You are a food identification assistant. Be specific and detailed.

Look at this image and answer:
1. TYPE: Is this (A) raw/uncooked ingredients laid out, or (B) a completed cooked dish?
   If you cannot tell, answer UNKNOWN.
2. DESCRIPTION: List every ingredient or food item you can see, separated by commas.
   Be specific — name the pasta shape, type of vegetable, type of cheese etc.
3. CONFIDENCE: A number from 0.0 to 1.0.

Reply in exactly this format (no extra text):
TYPE: A
DESCRIPTION: rigatoni pasta, canned crushed tomatoes, olive oil, parmesan cheese, white onion, garlic cloves, fresh basil, dried oregano, salt, black pepper
CONFIDENCE: 0.95
"""

class VisionProcessor:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f" Loading on {self.device}...")

        self.clip_model = CLIPModel.from_pretrained(clip_model_id)
        self.clip_model.to(self.device)
        self.clip_model.eval()
        #self.clip_model.requires_grad_(False)

        self.clip_proc  = CLIPProcessor.from_pretrained(clip_model_id)

        self.vlm_model = AutoModelForImageTextToText.from_pretrained(
            vlm_model_id,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto",
            ignore_mismatched_sizes=True,
        )
        self.vlm_processor = AutoProcessor.from_pretrained(vlm_model_id)
        print("VisionProcessor is initialized.")

    def encode_image(self, image: PILImage.Image) -> np.ndarray:
        inputs = self.clip_proc(images=image, return_tensors="pt")
        inputs = inputs.to(self.device)

        with torch.no_grad():
            embedding = self.clip_model.get_image_features(**inputs)
        embedding = embedding / embedding.norm(dim=-1, keepdim=True)
        embedding = embedding.cpu().numpy().astype("float32")
        embedding = embedding.flatten()
        return embedding

    def classify_image(self, image: PILImage.Image) -> dict:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": vlm_system_prompt},
            ],
        }]
        text_in = self.vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        img_inputs, _ = process_vision_info(messages)
        inputs = self.vlm_processor(text=[text_in], images=img_inputs,
                               return_tensors="pt").to(self.device)
        with torch.no_grad():
            out_ids = self.vlm_model.generate(**inputs, max_new_tokens=150)
        trimmed = out_ids[:, inputs["input_ids"].shape[1]:]
        raw = self.vlm_processor.batch_decode(trimmed, skip_special_tokens=True)[0]
        parsed_raw = self.parse_raw(raw)
        return parsed_raw

    def parse_raw(self, raw: str) -> dict:
        return self.parse(raw)

    def parse(self, text: str) -> dict:
        extracted_recipe_info = {
            "type": "unknown",
            "description": "",
            "confidence": 0.0
            }
        type_match = re.search(r"TYPE:\s*([AB]|UNKNOWN)", text, re.IGNORECASE)
        description = re.search(r"DESCRIPTION:\s*(.+)", text)
        conf = re.search(r"CONFIDENCE:\s*([\d.]+)", text)
        if type_match:
            t = type_match.group(1).upper()
            extracted_recipe_info["type"] = "ingredients" if t == "A" else "dish" if t == "B" else "unknown"
        if description:
          extracted_recipe_info["description"] = description.group(1).strip()
        if conf:
          extracted_recipe_info["confidence"] = float(conf.group(1))

        return extracted_recipe_info

vision_processor = VisionProcessor()
print("VisionProcessor is ready.")

 Loading on cuda...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

VisionProcessor is initialized.
VisionProcessor is ready.


## 6. CLIP Text Encoder
Encodes the VLM description (or raw text query) into the same 512-dim space as the recipe index.

In [6]:
class TextEncoder:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model = CLIPModel.from_pretrained(clip_model_id)
        self.clip_model.to(self.device)
        self.clip_model.eval()
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_id)
        print("TextEncoder is initialized.")

    def encode(self, text: str) -> np.ndarray:
        inputs = self.clip_processor(text=[text], return_tensors="pt",
                           truncation=True, max_length=77, padding=True)
        inputs = inputs.to(self.device)

        with torch.no_grad():
            embedding = self.clip_model.get_text_features(**inputs)
        # get_text_features returns a tensor directly, but if it returns an object, extract it
        if hasattr(embedding, "pooler_output"):
            embedding = embedding.pooler_output
        elif hasattr(embedding, "last_hidden_state"):
            embedding = embedding.last_hidden_state[:, 0, :]
        embedding = embedding / embedding.norm(dim=-1, keepdim=True)
        embedding = embedding.cpu().numpy().astype("float32")
        embedding = embedding.flatten()

        return embedding

    def encode_batch(self, texts: list) -> np.ndarray:
      inputs = self.clip_processor(text=texts, return_tensors="pt",
                        truncation=True, max_length=77, padding=True)
      inputs = inputs.to(self.device)

      with torch.no_grad():
          embedding = self.clip_model.get_text_features(**inputs)
      if hasattr(embedding, "pooler_output"):
          embedding = embedding.pooler_output
      elif hasattr(embedding, "last_hidden_state"):
          embedding = embedding.last_hidden_state[:, 0, :]
      embedding = embedding / embedding.norm(dim=-1, keepdim=True)
      embedding = embedding.cpu().numpy().astype("float32")
      return embedding

text_encoder = TextEncoder()
print("TextEncoder is ready.")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TextEncoder is initialized.
TextEncoder is ready.


## 7. Retrieval — FAISS + Web Search Fallback

In [7]:
import json
import faiss as _faiss
from tavily import TavilyClient

class FAISSRetriever:
    def __init__(self):
        print(f"Loading index from {faiss_index_local_path}...")
        self.index = _faiss.read_index(faiss_index_local_path)
        with open(recipes_metdata_local_path) as f:
            self.metadata = json.load(f)
        print(f"{self.index.ntotal:,} recipes loaded.")
        print("FAISSRetriever is initalized.")

    def search(self, embedding: np.ndarray, k: int = top_k):
        q = embedding.reshape(1, -1).astype("float32")
        scores, indices = self.index.search(q, k)
        recipes = []

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
              continue
            recipe = self.metadata[idx].copy()
            recipe["similarity_score"] = float(score)
            recipes.append(recipe)

        if len(scores[0]):
            best = float(scores[0][0])
        else:
            best = 0.0

        return recipes, best

class WebSearchRetriever:
    def __init__(self):
        self.client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
        print("WebSearchRetriever is initalized.")

    def search(self, query: str, k: int = top_k):
        web_response = self.client.search(
            query=f"recipe for {query}",
            search_depth="advanced",
            max_results=k,
            include_answer=True,
        )

        search_results = []
        for result_item in web_response.get("results", []):
            search_results.append({
                "title": result_item.get("title", "Web result"),
                "ingredients": [],
                "steps": [],
                "url": result_item.get("url", ""),
                "raw_content": result_item.get("content", ""),
                "similarity_score": None,
            })
        return search_results

faiss_retriever = FAISSRetriever()
web_retriever = WebSearchRetriever()
print("Retrievers are ready.")

Loading index from data/recipes.index...
1,000,000 recipes loaded.
FAISSRetriever is initalized.
WebSearchRetriever is initalized.
Retrievers are ready.


## 8. Synthesis — LLM Recipe Generator

In [8]:
import json as _json
from groq import Groq as _Groq

# synthesizer_llm_system_prompt = """You are a professional recipe writer and chef.
# Given a user query and retrieved recipe candidates, synthesise one perfect structured recipe.
# If the user provides a list of ingredients, create a recipe that uses THOSE SPECIFIC ingredients.
# If candidates are not relevant, use your own knowledge to create the best recipe.

# Return ONLY valid JSON matching this schema — no other text:
# {
  # "title": "...",
  # "description": "One sentence.",
  # "servings": 4,
  # "prep_time_minutes": 15,
  # "cook_time_minutes": 30,
  # "ingredients": [{ "item": "...", "quantity": "..." }],
  # "steps": [{ "step": 1, "instruction": "..." }],
  # "tips": ["..."],
  # "tags": ["..."]
# }"""

synthesis_llm_system_prompt = """You are a professional recipe writer and chef.
Given a user query and retrieved recipe candidates, synthesise one perfect structured recipe.
If the user provides a list of ingredients, create a recipe that uses THOSE SPECIFIC ingredients.
If candidates are not relevant, use your own knowledge to create the best recipe.

**STRICT VISUAL CONSTRAINT:** Make a recipe using the VLM identified ingredients [vlm_description].

First, think step by step inside <think>...</think> tags:
- Which candidate best matches the query and why?
- Are the ingredients consistent with the dish the user asked for?
- What adjustments should be made?

Then output ONLY valid JSON after the closing </think> tag, matching this schema:
{
  "title": "...",
  "description": "One sentence.",
  "ingredients": [{ "item": "...", "quantity": "..." }],
  "steps": [{ "step": 1, "instruction": "..." }],
  "tips": ["..."],
}"""

class SynthesisLLM:
    def __init__(self):
        self.client = _Groq(api_key=os.environ["GROQ_API_KEY"])
        print("SynthesisLLM is initialized.")

    def synthesise(self, query: str, candidates: list, image_type=None) -> dict:
        context = ""
        for i, r in enumerate(candidates[:5], 1):
            context += f"\n--- Candidate {i}: {r.get('title','Untitled')} ---\n"
            if r.get("raw_content"):
                context += r["raw_content"][:800] + "\n"
            else:
                if r.get("ingredients"):
                    context += "Ingredients: " + ", ".join(str(x) for x in r["ingredients"][:20]) + "\n"
                if r.get("steps"):
                    steps = r["steps"] if isinstance(r["steps"], list) else [r["steps"]]
                    context += "Steps: " + " ".join(str(s) for s in steps[:5]) + "\n"

        type_hint = ""
        if image_type == "ingredients":
            type_hint = "The user has raw ingredients and wants a recipe using them."
        elif image_type == "dish":
            type_hint = "The user has a photo of a completed dish and wants to recreate it."

        user_msg = f"User query: {query}\n{type_hint}\n\nCandidates:{context}\n\nSynthesise the best recipe."

        response = self.client.chat.completions.create(
            model=synthesis_model_id,
            messages=[
                {"role": "system", "content": synthesis_llm_system_prompt},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.4,
        )

        raw = response.choices[0].message.content

        # print chain of thought if present
        think_match = re.search(r"<think>(.*?)</think>", raw, re.DOTALL)
        if think_match:
            print("\n── LLM Chain of Thought ──────────────────────────────")
            print(think_match.group(1).strip())
            print("──────────────────────────────────────────────────────\n")

        # Strip think tags and parse JSON
        raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
        raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return _json.loads(raw)

synthesis_llm = SynthesisLLM()
print("SynthesisLLM is ready.")

SynthesisLLM is initialized.
SynthesisLLM is ready.


## 9. LangGraph Agent — Nodes & Graph

In [9]:
import functools
from langgraph.graph import StateGraph, END
import hashlib
import spacy

# Load a small English model from SpaCy
nlp = spacy.load("en_core_web_sm")

def get_nouns(text):
    doc = nlp(text.lower())
    # Extract only Nouns (NOUN) and Proper Nouns (PROPN)
    return [token.text for token in doc if token.pos_ in ["NOUN", "PROPN"] and len(token.text) > 2]

# All Tool/Node functions

# The vision_node identifies and obtain the ingredients and update the agent state accordingly
def vision_node(state: AgentState) -> AgentState:
    image = state.get("image")
    if image is None:
        return {
            **state,
            "image_type": None,
            "vlm_description": state.get("text_query", ""),
            "vlm_confidence": 1.0
        }

    result = vision_processor.classify_image(image)
    return {
        **state,
        "image_type": result["type"],
        "vlm_description": result["description"],
        "vlm_confidence": result["confidence"],
        "messages": state["messages"] + [{
            "role": "system",
            "content": f"'{result['type']}': {result['description']} (conf={result['confidence']:.2f})"
        }],
    }

# If the vision_node fails to identify and obtain the ingredients, then route to fallback and update the agent state accordingly
def fallback_node(state: AgentState) -> AgentState:
    count = state.get("fallback_count", 0) + 1
    msg = ("I couldn't confidently identify your image. "
           "Could you describe what you're cooking or what ingredients you have?")
    return {
        **state,
        "fallback_count": count,
        "image_type": "unknown",
        "vlm_description": None,
        "vlm_confidence": 0.0,
        "messages": state["messages"] + [{"role": "assistant", "content": msg}]
    }

def encode_query_node(state: AgentState) -> AgentState:
    vlm_description = state.get("vlm_description") or ""
    text_query = state.get("text_query") or ""

    noun_list = get_nouns(f"{text_query} {vlm_description}")
    unique_nouns = list(set(noun_list))

    ingredients_formatted = ", ".join(unique_nouns)
    clean_query = f"{text_query}: {ingredients_formatted}"

    embedding = text_encoder.encode(clean_query)

    return {
        **state,
        "query_embedding": embedding.tolist(),
        "combined_query": clean_query,
        "intent_words": unique_nouns
    }


def retrieval_node(state: AgentState) -> AgentState:

    embedding = np.array(state["query_embedding"], dtype="float32")
    recipes, best_score = faiss_retriever.search(embedding)

    # Using vlm_description (always contains actual food words the VLM extracted from the image) for relevance check instead of text_query,
    # because using text_query alone fails for vague and general queries like "give me a recipe for this" (because intent_words ends up empty —
    # no food words to match against FAISS titles)

    # query_intent = (state.get("text_query") or "").lower()

    # But, this replacement logic kept noisy words like 'make', 'using', or 'photo'. Instead use the high-quality Nouns already stored in the state from encode_query_node
    # query_intent = (state.get("vlm_description") or state.get("text_query") or "").lower()
    # intent_words = [w for w in query_intent.split() if len(w) > 3]

    intent_words = state.get("intent_words", [])

    def is_relevant(recipe):
        title = recipe.get("title", "").lower()
        # Look in ingredients too to avoid false-negative web searches
        ingredients = " ".join(recipe.get("ingredients", [])).lower()
        return any(word in title or word in ingredients for word in intent_words)

    faiss_relevant = any(is_relevant(r) for r in recipes)

    if best_score >= similarity_threshold and faiss_relevant:
        source = "faiss"
        print(f"FAISS score {best_score:.3f}, candidates relevant")
    else:
        # Fixed the print statement variable name
        reason = "below threshold" if best_score < similarity_threshold else "not relevant"
        print(f"FAISS score {best_score:.3f} {reason} for {intent_words}, doing web search")

        # Use full text query for web search context, fall back to clean query
        search_query = state.get("text_query") or state.get("combined_query")
        recipes = web_retriever.search(search_query)
        source = "web" if recipes else "none"

        # For new web results, ingest web results into FAISS and update
        if source == "web" and recipes:

            # creates hash title + first 100 chars of content as the unique content identifier
            def content_hash(title: str, content: str) -> str:
                return hashlib.md5(f"{title[:50]}:{content[:100]}".encode()).hexdigest()

            # existing_urls = {r.get("url", "") for r in faiss_retriever.metadata}
            existing_urls = {r.get("url", "") for r in faiss_retriever.metadata} # avoid duplicate contents using content hash, not just URL (because URL-only dedup fails when the same recipe appears under a different URL or when the same URL returns slightly different scraped content across queries)
            existing_hashes = {
                content_hash(r.get("title", ""), "".join(r.get("steps", [])) if isinstance(r.get("steps"), list) else r.get("raw_content", ""))
                for r in faiss_retriever.metadata
            }

            new_texts = []
            new_meta = []

            for r in recipes:
                url = r.get("url", "")
                chash = content_hash(r.get("title", ""), r.get("raw_content", ""))

                # Skip if duplicate by URL or content hash
                if (url and url in existing_urls) or chash in existing_hashes:
                    print(f" Skipping duplicate: {r.get('title', url)}")
                    continue

                title = r.get("title", "")
                content = r.get("raw_content", "")[:300]
                new_texts.append(f"{title}: {content}")
                new_meta.append({
                    "title": title,
                    "ingredients": r.get("ingredients", []),
                    "steps": r.get("steps", []),
                    "url": url,
                    "raw_content": r.get("raw_content", ""),  # kept for future hash checks
                })

            if new_texts and faiss_retriever.index.ntotal < max_index_size:
                slots_left = max_index_size - faiss_retriever.index.ntotal
                new_texts = new_texts[:slots_left]
                new_meta  = new_meta[:slots_left]

                new_embeddings = text_encoder.encode_batch(new_texts)
                faiss_retriever.index.add(new_embeddings)
                faiss_retriever.metadata.extend(new_meta)

                # Persist to disk so knowledge is stored even during session restarts
                faiss.write_index(faiss_retriever.index, faiss_index_local_path)
                with open(recipes_metdata_local_path, "w") as f_out:
                    json.dump(faiss_retriever.metadata, f_out)

                print(f"Added {len(new_texts)} new recipes. Index now has {faiss_retriever.index.ntotal:,} recipes.")

            elif faiss_retriever.index.ntotal >= max_index_size:
                print(f" Index at capacity ({max_index_size:,}), skipping ingestion.")

    return {
        **state,
        "retrieved_recipes": recipes,
        "retrieval_source": source,
        "messages": state["messages"] + [{
            "role": "system",
            "content": f" source={source}, {len(recipes)} candidates, faiss_relevant={faiss_relevant}"
        }]
    }


def synthesis_node(state: AgentState) -> AgentState:
    # Always use the combined query so Llama knows BOTH intent and ingredients
    #query  = state.get("combined_query") or state.get("text_query") or state.get("vlm_description", "")
    query = state.get("text_query") or state.get("combined_query")
    recipe = synthesis_llm.synthesise(query, state.get("retrieved_recipes") or [], state.get("image_type"))
    return {
        **state,
        "final_recipe": recipe,
        "messages": state["messages"] + [{
            "role": "assistant",
            "content": f"Recipe ready: {recipe.get('title', '')}"
        }]
    }

# Routing function to decide which tool/node to call next based on the agent state variables
def route_after_vision(state: AgentState) -> str:
    if state.get("image_type") is None:
      return "encode_query"
    if state.get("image_type") in ("ingredients", "dish") and state.get("vlm_confidence", 0) >= 0.5:
       return "encode_query"
    if state.get("fallback_count", 0) >= max_fallbacks:
      return "encode_query"

    return "fallback"

# Building the agent state graph
graph = StateGraph(AgentState)
graph.add_node("vision", vision_node)
graph.add_node("fallback", fallback_node)
graph.add_node("encode_query", encode_query_node)
graph.add_node("retrieval", retrieval_node)
graph.add_node("synthesis", synthesis_node)

graph.set_entry_point("vision")
graph.add_conditional_edges("vision", route_after_vision,
                             {"encode_query": "encode_query", "fallback": "fallback"})
graph.add_edge("fallback", END)
graph.add_edge("encode_query", "retrieval")
graph.add_edge("retrieval", "synthesis")
graph.add_edge("synthesis", END)

app = graph.compile()
print("LangGraph agent compiled.")

LangGraph agent compiled.


## 10. Helper — Pretty-print recipe

In [10]:
def pretty_print_recipe(recipe: dict):
    print("\n" + "═"*60)
    print(f"  {recipe.get('title','Recipe').upper()}")
    print("═"*60)
    print(f"  {recipe.get('description','')}")
    # print(f"\n  Serves {recipe.get('servings','?')}  |  "
          # f"Prep {recipe.get('prep_time_minutes','?')} min  |  "
          # f"Cook {recipe.get('cook_time_minutes','?')} min")
    print("\n  INGREDIENTS\n  " + "─"*40)
    for ing in recipe.get("ingredients", []):
        print(f" * {ing.get('quantity','')}  {ing.get('item','')}")
    print("\n  METHOD\n  " + "─"*40)
    for step in recipe.get("steps", []):
        print(f"  {step.get('step','')}. {step.get('instruction','')}")
    tips = recipe.get("tips", [])
    if tips:
        print("\n  TIPS")
        for tip in tips: print(f" * {tip}")
    print("═"*60 + "\n")

## 11. Run the Agent
Choose one of the three options below — text query, image path, or both.


In [14]:
from PIL import Image as PILImage

#text_query  = "Give me a recipe based on these ingredients"
text_query  = "How to make fried chicken?"
img_path  = "ingredients2.jpg"
image = PILImage.open(img_path).convert("RGB") if img_path else None

initial_state: AgentState = {
    "image": image,
    "text_query": text_query,
    "combined_query": None,
    "intent_words": None,
    "image_type": None,
    "vlm_description": None,
    "vlm_confidence": None,
    "query_embedding": None,
    "retrieved_recipes": None,
    "retrieval_source": None,
    "final_recipe": None,
    "messages": [],
    "fallback_count": 0,
    "error": None,
}

final_state = app.invoke(initial_state)

# Handle fallback (agent asked for clarification)
if final_state.get("image_type") == "unknown" and not final_state.get("final_recipe"):
    last_msg = final_state["messages"][-1]["content"]
    print(f"\n[Agent] {last_msg}")
    user_input = input("Your answer: ").strip()
    final_state["text_query"] = user_input
    final_state["image"] = None
    final_state = app.invoke(final_state)

recipe = final_state.get("final_recipe")
if recipe:
    pretty_print_recipe(recipe)
else:
    print("\n[Agent] Sorry, I couldn't generate a recipe. Please try again.")

FAISS score 0.688 not relevant for ['pepper', 'paprika', 'salt', 'milk', 'oil', 'sugar', 'spices', 'powder', 'flour', 'drumsticks', 'chicken'], doing web search
Added 5 new recipes. Index now has 1,000,005 recipes.

── LLM Chain of Thought ──────────────────────────────
To synthesize the best recipe for fried chicken based on the provided candidates and the user's query, we need to consider the ingredients and steps that are most commonly associated with traditional fried chicken recipes. 

Candidate 1 provides a detailed breading process, which is crucial for achieving crispy fried chicken. However, it includes ingredients like eggs, hot sauce, and Worcestershire sauce, which might not be essential for a basic fried chicken recipe.

Candidate 2 involves a brine and a seasoning mix that includes a variety of spices, which could enhance the flavor of the chicken. It also mentions creating a wet batter, which is an interesting approach but might not be necessary for a simple fried chicke

## 12. Inspect Intermediate State
Optional — useful for debugging.

In [15]:
print("image_type: ", final_state.get("image_type"))
print("vlm_description: ", final_state.get("vlm_description"))
print("vlm_confidence: ", final_state.get("vlm_confidence"))
print("retrieval_source: ", final_state.get("retrieval_source"))
print("candidates found: ", len(final_state.get("retrieved_recipes") or []))
print("Conversation log:")
for m in final_state.get("messages", []):
    print(f" [{m['role']}] {m['content']}")

image_type:  ingredients
vlm_description:  chicken drumsticks, flour, sugar, salt, pepper, paprika, milk, oil, spices, baking powder
vlm_confidence:  0.98
retrieval_source:  web
candidates found:  5
Conversation log:
 [system] 'ingredients': chicken drumsticks, flour, sugar, salt, pepper, paprika, milk, oil, spices, baking powder (conf=0.98)
 [system]  source=web, 5 candidates, faiss_relevant=False
 [assistant] Recipe ready: Simple Fried Chicken Recipe


In [13]:
import zipfile

with zipfile.ZipFile('data.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk('data'):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, 'data'))
print(f"zip complete")

zip complete


System Deliverables

State Management: You're correctly using the AgentState to pass information (image, VLM description, query embedding, retrieved recipes) between different processing steps.
Modular Design: Each component (vision, encoding, retrieval, synthesis) is encapsulated in its own node, making the logic clear and maintainable.
Conditional Routing: The route_after_vision function demonstrates a key LangGraph feature, allowing the flow to adapt based on the VLM's confidence and image classification. This handles a basic error/fallback scenario gracefully.
Directed Acyclic Graph (DAG) for a fixed workflow: For a process that generally follows a sequence but has some branching (like your fallback), LangGraph is excellent.

Future Enhancements

Potential for Further Exploration of Full Potential:

Conversational Loops/Human-in-the-Loop within the Graph: Currently, the fallback loop where the agent asks for user input (if final_state.get("image_type") == "unknown"...) happens outside the compiled LangGraph app. LangGraph excels at building true conversational agents where the graph itself can loop back to a 'human input' node, allowing the agent to ask clarifying questions and re-process based on user feedback within the graph's execution flow. This would make the agent more interactive and robust for multi-turn conversations.
Dynamic Tool Use/Agentic Behavior: While your nodes are effectively tools, LangGraph can integrate with LangChain tools, allowing the agent to dynamically decide which tool to use next, or to chain multiple tools based on the ongoing state. This could be relevant if you wanted more complex reasoning steps or external API calls.
Error Handling and Retries: Beyond a simple fallback, you could define more sophisticated error handling nodes or retry mechanisms within the graph for specific failures (e.g., if a web search fails, try a different search strategy).
Parallel Processing: For certain tasks, if parts of your workflow could run in parallel, LangGraph can support that by running multiple nodes concurrently.
In summary, for orchestrating a multi-step, conditionally branched pipeline like this recipe generator, your current use of LangGraph is solid and effective. If you wanted to expand this into a more conversational, adaptive, or complex problem-solving agent, then you'd dive deeper into features like internal loops, dynamic tool calling, and more elaborate human-in-the-loop patterns.

TODO: There is a data disconnect between the VLM and the synthesis LLM for sharing VLM description:

When verifying the VLM desc of ingredients2.jpg in cell 12:

image_type:  ingredients
vlm_description:  chicken drumsticks, flour, sugar, salt, pepper, paprika, milk, oil, spices, baking powder
vlm_confidence:  0.98
retrieval_source:  web
candidates found:  5
Conversation log:
 [system] 'ingredients': chicken drumsticks, flour, sugar, salt, pepper, paprika, milk, oil, spices, baking powder (conf=0.98)
 [system]  source=web, 5 candidates, faiss_relevant=False
 [assistant] Recipe ready: Simple Fried Chicken Recipe



 FAISS score 0.688 not relevant for ['pepper', 'paprika', 'salt', 'milk', 'oil', 'sugar', 'spices', 'powder', 'flour', 'drumsticks', 'chicken'], doing web search
Added 5 new recipes. Index now has 1,000,005 recipes.

── LLM Chain of Thought ──────────────────────────────
To synthesize the best recipe for fried chicken based on the provided candidates and the user's query, we need to consider the ingredients and steps that are most commonly associated with traditional fried chicken recipes.

Candidate 1 provides a detailed breading process, which is crucial for achieving crispy fried chicken. However, it includes ingredients like eggs, hot sauce, and Worcestershire sauce, which might not be essential for a basic fried chicken recipe.

Candidate 2 involves a brine and a seasoning mix that includes a variety of spices, which could enhance the flavor of the chicken. It also mentions creating a wet batter, which is an interesting approach but might not be necessary for a simple fried chicken recipe.

Candidate 3, Chicken-Fried Chicken, seems to focus more on the method of pounding the chicken thin and then frying it, which is slightly different from traditional fried chicken. It also includes a recipe for cream gravy, which is not directly related to the frying process.

Candidate 4 provides a straightforward approach to frying chicken, emphasizing the importance of not overcrowding the pan and turning the chicken regularly. However, it lacks detail in the breading process.

Candidate 5 offers specific temperatures and times for frying different parts of the chicken, which is very helpful for achieving the right crispiness and doneness. It also mentions the importance of letting the crust bond with the chicken skin before moving it, which is a valuable tip.

Considering the user's query is about making fried chicken without specifying any particular ingredients or methods, we can synthesize a recipe that combines the most essential and universally applicable steps and ingredients from these candidates.

The key ingredients for a basic fried chicken recipe would include chicken pieces, flour, spices (like paprika, garlic powder, salt, and pepper), and oil for frying. The process would involve seasoning the chicken, dredging it in flour, and then frying it in hot oil until it's golden brown and cooked through.

Given the strict visual constraint and the need to prioritize ingredients identified by the VLM description (which are not explicitly provided in this scenario), we will focus on creating a recipe that uses basic ingredients commonly found in fried chicken recipes and avoids adding major proteins or primary ingredients not explicitly listed.

Adjustments should be made to ensure the recipe is simple, straightforward, and uses minimal ingredients while still achieving the crispy exterior and juicy interior characteristic of good fried chicken.
──────────────────────────────────────────────────────


════════════════════════════════════════════════════════════
  SIMPLE FRIED CHICKEN RECIPE
════════════════════════════════════════════════════════════
  A basic recipe for fried chicken that yields crispy exterior and juicy interior.

  INGREDIENTS
  ────────────────────────────────────────
 * 2-3 pounds  Chicken pieces (legs, thighs, wings, breasts)
 * 1 cup  All-purpose flour
 * 1 teaspoon  Paprika
 * 1 teaspoon  Garlic powder
 * 1 teaspoon  Salt
 * 1/2 teaspoon  Black pepper
 * about 1/2-inch deep in a frying pan  Vegetable oil

  METHOD
  ────────────────────────────────────────
  1. In a shallow dish, mix together the flour, paprika, garlic powder, salt, and pepper.
  2. Dredge each piece of chicken in the flour mixture, coating both sides evenly.
  3. Heat about 1/2-inch of vegetable oil in a large skillet over medium-high heat until it reaches 350°F.
  4. Carefully place 2-3 pieces of the coated chicken into the hot oil. Do not overcrowd the skillet.
  5. Fry for 5-7 minutes on each side or until the chicken is golden brown and the internal temperature reaches 165°F. Repeat with the remaining chicken pieces.
  6. Remove the fried chicken from the oil with a slotted spoon and place it on a paper towel-lined plate to drain excess oil. Serve hot.

  TIPS
 * Ensure the oil is at the right temperature before frying to prevent the chicken from absorbing too much oil.
 * Do not overcrowd the skillet to allow for even cooking and to prevent the oil temperature from dropping.
 * Let the chicken rest for a few minutes before serving to allow the juices to redistribute.
════════════════════════════════════════════════════════════


